# IndexTTS 2.5（现代 Web 语音合成工作台）

- **支持显卡**：A100 (40GB / 80GB)、H100 (80GB)、RTX 6000 (96GB)
- **底层引擎**：官方原生 `index-tts`（零 ComfyUI 依赖，BF16 推理，含 QwenEmotion）
- **单页多合一能力**：零样本多语种克隆（`zh / en / ja / es / ar`）、语速调节、自然语言情绪描述或情绪参考音频、试听与会话历史回溯
- **访问方式**：使用 Google Colab 官方内置端口代理（`proxyPort`）在浏览器直接打开工作台，绑定当前 Google 账号鉴权，无需外部隧道或密码，不开放外部 API
- **执行顺序**：`1. 环境检测` → `2. 安装 IndexTTS 2.5 引擎与 Web 工作台` → `3. 下载 IndexTTS 2.5 模型权重` → `4. 启动 Web 工作台`

### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存、磁盘空间及 PyTorch / CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`，显存 40GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 1. Environment Check {display-mode: "form"}
import os, shutil, subprocess, torch

MIN_VRAM_GIB = 35
REQUIREMENT = "40GB+ VRAM (A100 40GB/80GB, H100 80GB, RTX 6000 96GB)"

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select Runtime -> Change runtime type -> A100 / H100 / RTX 6000 GPU.")

print(subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gib = shutil.disk_usage("/content").free / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu} | VRAM: {vram_gib:.1f} GiB | RAM: {ram_gib:.1f} GiB | Disk Free: {disk_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if not any(k in gpu.upper() for k in ("A100", "H100", "6000")) or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"Detected {gpu} ({vram_gib:.1f} GiB), but this notebook requires {REQUIREMENT}. Please switch runtime.")
if not bf16:
    raise RuntimeError("Current GPU does not support BF16; please switch runtime.")
print(f"[OK] GPU environment ready: {gpu} ({vram_gib:.1f} GiB)")

### 2. 安装 IndexTTS 2.5 引擎与 Web 工作台
- **功能**：安装官方 `index-tts` 推理引擎及 Web 工作台依赖，并拉取校验 GitHub Release 发布的 `app-indextts-2.5.tar.gz`（`v0.3.0`，内含编译好的现代前端 `dist` 与本地 Web 后端）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 2. Install IndexTTS 2.5 Engine & App Bundle {display-mode: "form"}
import hashlib, shutil, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

INDEXTTS_DIR = Path("/content/index-tts")
INDEXTTS_REPO = "https://github.com/index-tts/index-tts"
INDEXTTS_REVISION = "d9e41aac89fd00b3d71497fddb287b7f24613712"

APP_DIR = Path("/content/app")
RELEASE_REPO = "boltguo/colab"
RELEASE_TAG = "v0.3.0"
APP_ARCHIVE = "app-indextts-2.5.tar.gz"
APP_SHA256 = "e877060da4a4763c2f5b0dd94e3836a90755d58a8e726a768801f73dadde962c"

RUNTIME_PACKAGES = [
    "transformers==4.52.1",
    "tokenizers>=0.21.0",
    "accelerate>=1.8.0",
    "safetensors>=0.4.3",
    "huggingface_hub>=0.23.0",
    "hf_xet",
    "librosa>=0.10.2",
    "soundfile>=0.10.0",
    "scipy>=1.10.0",
    "omegaconf>=2.3.0",
    "einops>=0.8.1",
    "sentencepiece>=0.2.1",
    "tiktoken>=0.5.0",
    "openai-whisper>=20231117",
    "modelscope>=1.27.0",
    "munch>=4.0.0",
    "json5>=0.9.16",
    "textstat>=0.7.10",
    "jieba>=0.42.1",
    "cn2an>=0.5.22",
    "g2p-en>=2.1.0",
    "ffmpeg-python>=0.2.0",
    "descript-audiotools>=0.7.2",
    "PyYAML>=6.0",
    "wetext>=0.0.9",
    "fugashi>=1.2.0",
    "unidic-lite>=1.0.0",
    "fastapi>=0.115.0,<1.0.0",
    "uvicorn[standard]>=0.30.0,<1.0.0",
    "python-multipart>=0.0.9,<1.0.0",
    "pydantic>=2.7.0,<3.0.0",
]

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def git(path, *args):
    return subprocess.check_output(["git", *args], cwd=path, text=True).strip()

def ensure_not_symlink(path, label):
    for p in [path, *path.parents]:
        if p == Path("/content"):
            break
        if p.is_symlink():
            raise RuntimeError(f"{label} path contains a symlink; start a fresh runtime: {p}")

def checkout_repo(path, repo_url, revision, label, allow_local_changes=False):
    ensure_not_symlink(path, label)
    fresh = not path.exists()
    if fresh:
        path.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "--no-checkout", repo_url + ".git", str(path)])
    if not (path / ".git").is_dir():
        raise RuntimeError(f"{label} is not a git repository; please use a clean runtime: {path}")
    if git(path, "remote", "get-url", "origin").rstrip("/").removesuffix(".git") != repo_url:
        raise RuntimeError(f"Unexpected remote origin in {label}; please use a clean runtime.")
    if not fresh and not allow_local_changes and git(path, "status", "--porcelain"):
        raise RuntimeError(f"{label} has uncommitted local changes; please use a clean runtime.")
    if fresh or git(path, "rev-parse", "HEAD") != revision:
        run(["git", "fetch", "--depth", "1", "origin", revision], cwd=path)
        run(["git", "checkout", "--force", "--detach", revision], cwd=path)

def install_release_app():
    ensure_not_symlink(APP_DIR, "App directory")
    stamp_file = APP_DIR / ".release_sha256"
    if (
        stamp_file.is_file()
        and not stamp_file.is_symlink()
        and stamp_file.read_text(encoding="utf-8").strip() == f"{RELEASE_TAG}:{APP_SHA256}"
        and (APP_DIR / "backend/run_server.py").is_file()
    ):
        print(f"App bundle already up to date: {APP_ARCHIVE} ({RELEASE_TAG})")
        return

    local_archive = Path(f"/content/{APP_ARCHIVE}")
    with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        if local_archive.is_file() and not local_archive.is_symlink():
            print(f"Using pre-staged app archive: {local_archive}")
            payload = local_archive.read_bytes()
        else:
            url = f"https://github.com/{RELEASE_REPO}/releases/download/{RELEASE_TAG}/{APP_ARCHIVE}"
            print(f"Downloading app bundle: {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "colab-release-fetcher"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                payload = resp.read()
        actual_sha = hashlib.sha256(payload).hexdigest()
        if actual_sha != APP_SHA256:
            raise RuntimeError(
                f"App bundle SHA256 mismatch! Expected {APP_SHA256}, got {actual_sha}"
            )
        tmp_path.write_bytes(payload)

        staging = APP_DIR.with_name("app.staging")
        if staging.is_symlink():
            raise RuntimeError("Staging directory is a symlink; aborting.")
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir(parents=True, exist_ok=True)
        staging_root = staging.resolve()

        with tarfile.open(tmp_path, "r:gz") as tf:
            for member in tf.getmembers():
                if member.issym() or member.islnk():
                    raise RuntimeError(f"Refusing symlink/hardlink archive entry: {member.name}")
                target = (staging / member.name).resolve()
                if target != staging_root and not str(target).startswith(str(staging_root) + "/"):
                    raise RuntimeError(f"Archive path escapes staging root: {member.name}")
            tf.extractall(staging)

        (staging / ".release_sha256").write_text(f"{RELEASE_TAG}:{APP_SHA256}\n", encoding="utf-8")
        if APP_DIR.exists():
            shutil.rmtree(APP_DIR)
        staging.replace(APP_DIR)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()

checkout_repo(INDEXTTS_DIR, INDEXTTS_REPO, INDEXTTS_REVISION, "index-tts", allow_local_changes=True)

# Use precompiled wetext normalizer on Linux to avoid building pynini from source.
front_py = INDEXTTS_DIR / "indextts/utils/front.py"
front_py.write_text(
    front_py.read_text(encoding="utf-8").replace('if platform.system() != "Linux":', "if True:"),
    encoding="utf-8",
)

run([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PACKAGES])
# Restore modern protobuf after descript-audiotools (which declares protobuf<3.20) for Colab Python 3.13 / TF.
run([sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf>=5.28.0,<7.0.0"])
install_release_app()

print(f"index-tts: {INDEXTTS_REVISION[:12]} | app: {RELEASE_TAG} ({APP_SHA256[:12]})")
print("[OK] IndexTTS 2.5 engine and Web Studio bundle installed (zero ComfyUI dependency).")

### 3. 下载 IndexTTS 2.5 模型权重
- **功能**：下载 IndexTTS-2.5 权重（含 QwenEmotion）与辅助编码器 / 声码器（约 5.1 GB）至 `/content/models/IndexTTS-2.5`，并预缓存辅助特征提取与声码器模型。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 3. Download IndexTTS 2.5 Model Weights {display-mode: "form"}
import os, shutil, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
from huggingface_hub import hf_hub_download

MODEL_DIR = Path("/content/models/IndexTTS-2.5")
INDEXTTS_DIR = Path("/content/index-tts")
INDEXTTS = ("IndexTeam/IndexTTS-2.5", "c39ce5ba981572cb187443877ff559dfb246ce63", "")
MODEL_FILES = [
    (*INDEXTTS, "config.yaml", 2860),
    (*INDEXTTS, "gpt.pth", 3259599833),
    (*INDEXTTS, "s2mel.pth", 414908601),
    (*INDEXTTS, "codec.pth", 607290935),
    (*INDEXTTS, "wav2vec2bert_stats.pt", 9343),
    (*INDEXTTS, "feat1.pt", 57170),
    (*INDEXTTS, "feat2.pt", 374866),
    (*INDEXTTS, "multilingual_zh_ja_yue_char_del.tiktoken", 907395),
    (*INDEXTTS, "qwen0.6bemo4-merge/Modelfile", 360),
    (*INDEXTTS, "qwen0.6bemo4-merge/added_tokens.json", 707),
    (*INDEXTTS, "qwen0.6bemo4-merge/chat_template.jinja", 550),
    (*INDEXTTS, "qwen0.6bemo4-merge/config.json", 727),
    (*INDEXTTS, "qwen0.6bemo4-merge/generation_config.json", 117),
    (*INDEXTTS, "qwen0.6bemo4-merge/merges.txt", 1671853),
    (*INDEXTTS, "qwen0.6bemo4-merge/model.safetensors", 1192135096),
    (*INDEXTTS, "qwen0.6bemo4-merge/special_tokens_map.json", 616),
    (*INDEXTTS, "qwen0.6bemo4-merge/tokenizer.json", 11422654),
    (*INDEXTTS, "qwen0.6bemo4-merge/tokenizer_config.json", 5433),
    (*INDEXTTS, "qwen0.6bemo4-merge/vocab.json", 2776833),
]

def local_directory(path):
    path = Path(path)
    for parent in [path, *path.parents]:
        if parent == Path("/content"):
            break
        if parent.is_symlink():
            raise RuntimeError(f"Directory is a symlink; please start a fresh runtime: {parent}")
    path.mkdir(parents=True, exist_ok=True)

def model_ready(target, size):
    return target.is_file() and not target.is_symlink() and target.stat().st_size == size

def download_model(repo, revision, _folder, file, size):
    target = MODEL_DIR / file
    local_directory(target.parent)
    started = time.time()
    result = Path(
        hf_hub_download(
            repo_id=repo,
            filename=file,
            revision=revision,
            local_dir=str(MODEL_DIR),
            force_download=target.exists(),
            token=False,
        )
    )
    if result.resolve() != target.resolve() or not model_ready(target, size):
        raise RuntimeError(f"Incomplete model download: {target}")
    print(f"Downloaded: {file} ({size / 1024**3:.2f} GiB in {time.time() - started:.0f}s)")

def download_models():
    local_directory(MODEL_DIR)
    pending = []
    for item in MODEL_FILES:
        target = MODEL_DIR / item[3]
        if target.is_symlink():
            raise RuntimeError(f"Refusing symlink model file: {target}")
        if model_ready(target, item[4]):
            print("Verified:", item[3])
        else:
            pending.append(item)
    if pending:
        missing = sum(item[4] for item in pending)
        free = shutil.disk_usage(MODEL_DIR).free
        if free < missing + 2 * 1024**3:
            raise RuntimeError(
                f"Insufficient disk space: need {missing / 1024**3:.1f} GiB more, free {free / 1024**3:.1f} GiB."
            )
        started = time.time()
        pool = ThreadPoolExecutor(max_workers=3)
        try:
            futures = [pool.submit(download_model, *item) for item in sorted(pending, key=lambda x: -x[4])]
            for future in as_completed(futures):
                future.result()
        finally:
            pool.shutdown(wait=False, cancel_futures=True)
        elapsed = time.time() - started
        print(
            f"Primary weights ready: {missing / 1024**3:.1f} GiB in {elapsed:.0f}s ({missing / 1024**2 / max(elapsed, 1):.0f} MB/s)"
        )

    if str(INDEXTTS_DIR) not in sys.path:
        sys.path.insert(0, str(INDEXTTS_DIR))
    from indextts.utils.model_download import ensure_models_available
    ensure_models_available(str(MODEL_DIR))

download_models()
print("[OK] IndexTTS 2.5 weights and auxiliary vocoders ready.")

### 4. 启动 Web 工作台（Colab 原生代理）
- **功能**：在本地回环地址（`127.0.0.1:8188`）启动 Web Studio，并通过 **Google Colab 原生端口代理** 生成当前浏览器的专属安全访问链接。
- **访问说明**：无需设置密码或配置第三方隧道，直接由当前登录的 Google 账号鉴权访问；不开放外部 API，实例停止后临时链接失效。
- **使用**：请在浏览器打开本 Colab 笔记本并直接运行下方单元格，完成后点击输出的链接打开工作台。

In [ ]:
# @title 4. Launch Web Studio {display-mode: "form"}
import json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path
from urllib.parse import urlsplit

FEATURE_NAME = "indextts-2.5"
APP_DIR = Path("/content/app")
DATA_DIR = Path(f"/content/app_data/{FEATURE_NAME}")
PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
LOG = Path(f"/content/{FEATURE_NAME}-web.log")
LAUNCH_CONFIG = Path(f"/content/{FEATURE_NAME}.launch.json")
PROCESS_PATTERN = r"backend/run_server\.py|backend\.run_server"

def read_text(path):
    return path.read_text(errors="replace") if path.exists() else ""

def tail(path, size=8000):
    return read_text(path)[-size:]

def rotate_log(path):
    if path.is_file() and path.stat().st_size:
        path.replace(path.with_name(path.name + ".1"))

def process_running(pattern):
    return subprocess.run(["pgrep", "-f", pattern], stdout=subprocess.DEVNULL).returncode == 0

def stop_processes(pattern, timeout):
    subprocess.run(["pkill", "-f", pattern], check=False)
    deadline = time.time() + timeout
    while process_running(pattern) and time.time() < deadline:
        time.sleep(2)
    if process_running(pattern):
        subprocess.run(["pkill", "-9", "-f", pattern], check=False)
        time.sleep(2)

def resolve_colab_proxy_url(port):
    try:
        from google.colab import output
    except ImportError:
        return f"http://127.0.0.1:{int(port)}", "local"
    try:
        proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({int(port)})")
    except Exception as exc:
        raise RuntimeError(
            "Failed to resolve Colab native proxy URL. Run Step 4 inside a browser Google Colab notebook session."
        ) from exc
    if not isinstance(proxy_url, str) or not proxy_url.strip().startswith("https://"):
        raise RuntimeError(
            f"Colab did not return a valid HTTPS proxy URL ({proxy_url!r}). Open the notebook in your browser and re-run Step 4."
        )
    return proxy_url.strip().rstrip("/"), "colab"

def open_colab_proxy(port, proxy_url, access_mode):
    if access_mode == "colab":
        from google.colab import output
        output.serve_kernel_port_as_window(int(port))
        print(f"[OK] Web Studio (Colab Native Proxy): {proxy_url}")
        print("Access is restricted to your signed-in Google account and expires when this runtime stops.")
    else:
        print(f"[OK] Web Studio (Local): {proxy_url}")

if APP_DIR.is_symlink() or not (APP_DIR / "backend/run_server.py").is_file():
    raise RuntimeError("App directory /content/app not found; please run Step 2 first.")

if FEATURE_NAME == "indextts-2.5":
    missing_media_tools = [name for name in ("ffmpeg", "ffprobe") if not shutil.which(name)]
    if missing_media_tools:
        raise RuntimeError(f"Missing required media tools: {', '.join(missing_media_tools)}. Run !apt-get -qq update && apt-get -qq install -y ffmpeg first.")

def ready(timeout=5):
    try:
        with urllib.request.urlopen(BASE + "/api/v1/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

proxy_url, access_mode = resolve_colab_proxy_url(PORT)
proxy_origin = f"{urlsplit(proxy_url).scheme}://{urlsplit(proxy_url).netloc}" if access_mode == "colab" else ""

env_overrides = {
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "COLAB_FEATURE": FEATURE_NAME,
    "COLAB_HOST": "127.0.0.1",
    "COLAB_PORT": str(PORT),
    "COLAB_DATA_DIR": str(DATA_DIR),
    "COLAB_DIST_DIR": str(APP_DIR / "dist" / FEATURE_NAME),
    "COLAB_PROXY_ORIGIN": proxy_origin,
    "INDEXTTS_SRC_DIR": "/content/index-tts",
    "INDEXTTS_MODEL_DIR": "/content/models/IndexTTS-2.5",
}

cmd = [
    sys.executable,
    str(APP_DIR / "backend/run_server.py"),
    "--feature",
    FEATURE_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--dist-dir",
    str(APP_DIR / "dist" / FEATURE_NAME),
]

launch_fingerprint = json.dumps({
    "cmd": cmd,
    "release": read_text(APP_DIR / ".release_sha256").strip(),
    "env": env_overrides,
}, sort_keys=True)
running = ready() or (process_running(PROCESS_PATTERN) and ready(timeout=15))
if running and read_text(LAUNCH_CONFIG) != launch_fingerprint:
    print("Service configuration changed; restarting Web Studio...")
    stop_processes(PROCESS_PATTERN, 30)
    running = False

if running:
    print("[OK] Web Studio server already running.")
else:
    stop_processes(PROCESS_PATTERN, 30)
    rotate_log(LOG)
    child_env = {**os.environ, **env_overrides}
    with LOG.open("w") as log:
        process = subprocess.Popen(
            cmd,
            cwd=APP_DIR,
            env=child_env,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    LAUNCH_CONFIG.write_text(launch_fingerprint, encoding="utf-8")
    print("Starting FastAPI Web Studio...")
    deadline = time.time() + 120
    while time.time() < deadline:
        if process.poll() is not None:
            print(tail(LOG))
            raise RuntimeError(f"App server failed to start (exit code {process.returncode}); see logs above.")
        if ready():
            print("[OK] Web Studio server started.")
            break
        time.sleep(1.5)
    else:
        print(tail(LOG))
        raise TimeoutError("App server did not become ready within 2 minutes; see logs above.")

if not DATA_DIR.is_symlink():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    proxy_info_path = DATA_DIR / "proxy_info.json"
    if not proxy_info_path.is_symlink():
        proxy_info_path.write_text(
            json.dumps({"public_url": proxy_url, "access_mode": access_mode}, ensure_ascii=False),
            encoding="utf-8",
        )
open_colab_proxy(PORT, proxy_url, access_mode)